In [0]:
USE CATALOG IDENTIFIER(:catalog);

In [0]:
CREATE OR REPLACE TABLE gold.orders_by_store
AS
SELECT e.StoreName, COUNT(DISTINCT order_id) total_orders
FROM silver.fact_orders f
INNER JOIN silver.store_employees e
  ON f.employee_id = e.EmployeeId
GROUP BY ALL
ORDER BY StoreName;

In [0]:
CREATE OR REPLACE TABLE gold.employees_per_store
AS
SELECT e.StoreName, COUNT(EmployeeId) total_orders
FROM silver.store_employees e  
WHERE IsActive
GROUP BY ALL
ORDER BY StoreName;

In [0]:
CREATE OR REPLACE TABLE gold.order_items_by_store
AS
WITH total_order_items
AS
(
  SELECT 
    order_id,
    order_amount,
    SUM(quantity) as total_items_ordered
  FROM silver.fact_orders
  GROUP BY ALL
),
cte_emp
AS
(
  SELECT 
    order_id,
    employee_id
  FROM silver.fact_orders
  GROUP BY ALL
)
SELECT t.*, se.StoreName
FROM total_order_items t
INNER JOIN cte_emp ce
  ON t.order_id = ce.order_id
INNER JOIN silver.store_employees se
  ON ce.employee_id = se.EmployeeId
  AND se.IsActive;

In [0]:
CREATE OR REPLACE TABLE gold.product_sales
AS
WITH ord_prd_cte
AS
(
  SELECT 
    order_id,
    product_id,    
    SUM(quantity) AS total_qty
  FROM silver.fact_orders fo
  GROUP BY ALL
)
SELECT 
  p.Name,
  p.Brand,
  p.Color,
  p.Category,
  SUM(op.total_qty) AS total_qty
FROM ord_prd_cte op
INNER JOIN silver.product p
  ON op.product_id = p.ProdId
GROUP BY ALL